In [1]:
# Load libraries
import os
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


ModuleNotFoundError: No module named 'scanpy'

In [3]:
!pip install scanpy umap-learn anndata numpy scipy pandas matplotlib seaborn python-igraph louvain leidenalg

^C


In [ ]:
# Set working directory
os.chdir("/media/hdd/nadeem/scRNAseq_workshop/")

In [ ]:
# Read sample information
sampleinfo = pd.read_csv("sampleinfo.csv")
sampleinfo

In [ ]:
# Read 10X data (per sample)
adatas = []

for sample in sampleinfo["SampleName"]:
    adata = sc.read_10x_mtx(
        path=sample,
        var_names="gene_symbols",
        cache=True
    )

    # Add metadata
    adata.obs["sample"] = sample

    adatas.append(adata)

In [ ]:
print(adatas)

In [ ]:
type(adatas)

In [ ]:
type(adata)

In [ ]:
# Check the shape of each AnnData object
for adata in adatas:
    sample_name = adata.obs["sample"].unique()[0]
    print(sample_name, adata.shape)


In [ ]:
# Extract the AnnData object for sample RZ1
adata_rz1 = [a for a in adatas if a.obs["sample"].unique()[0] == "RZ1"][0]
adata_rz1[:5, :5].X


In [ ]:
# Convert the sparse matrix to a DataFrame
adata_rz1[:5, :5].to_df()


In [ ]:
# Check the first few columns (var) of the obs DataFrame for RZ1
adata_rz1.var.head()

In [ ]:
# Check the first few rows (obs) of the obs DataFrame for RZ1
adata_rz1.obs.head()

In [ ]:
# Merge samples into one object
adata = ad.concat(
    adatas,
    join="outer",
    label="sample_id",
    keys=sampleinfo["SampleName"]
)

In [ ]:
print(adata)

In [ ]:
adata.X.shape

In [ ]:
(adata.X > 0).sum() / (adata.n_obs * adata.n_vars)

In [ ]:
adata.X.min(), adata.X.max()

In [ ]:
# Export raw cell metadata
adata.obs.to_csv("cell_metadata_raw.csv")

In [ ]:
### Quality control metrics ###
# Number of genes per cell (0, gene absent, non zero, gene present)
adata.obs["n_genes"] = (adata.X > 0).sum(axis=1).A1
adata.obs["n_genes"].head()

In [ ]:
# Compute basic QC metrics
# Total counts/UMIs per cell
adata.obs["n_counts"] = adata.X.sum(axis=1).A1 # Cell_1: gene1 + gene2 + ... + gene56941 = total UMIs counts
adata.obs["n_counts"].head()

In [ ]:
# Average counts per gene
adata.obs["counts_per_gene"] = adata.obs["n_counts"] / adata.obs["n_genes"]
adata.obs["counts_per_gene"].head()

In [ ]:
# Identify mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith(("MT-", "mt-"))

In [ ]:
# how many mitochondrial genes are there?
adata.var["mt"].sum()

In [ ]:
# Identify ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))


In [ ]:
# how many ribosomal genes are there?
adata.var["ribo"].sum()

In [ ]:
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo"],
    percent_top=None,
    log1p=False,
    inplace=True
)

In [ ]:
# Check head of obs DataFrame to see the new QC metrics
adata.obs.head()

In [ ]:
# export the QC metrics to a CSV file
qc_cols = [
    "sample", "n_counts", "n_genes", "counts_per_gene", "pct_counts_mt", "pct_counts_ribo"
]
adata.obs.to_csv("QC_raw_table.csv")

In [ ]:
# Histogram of total counts per cell
plt.figure(figsize=(6, 4))
sns.histplot(adata.obs["n_counts"], bins=50, kde=False)
plt.xlabel("Total Counts per Cell")
plt.ylabel("Number of Cells")
plt.title("Distribution of Total Counts per Cell")
plt.show()

# Histogram of number of genes per cell
plt.figure(figsize=(6, 4))
sns.histplot(adata.obs["n_genes"], bins=50, kde=False)
plt.xlabel("Number of Genes per Cell")
plt.ylabel("Number of Cells")
plt.title("Distribution of Number of Genes per Cell")
plt.show()

# Histogram of counts per gene
plt.figure(figsize=(6, 4))
sns.histplot(adata.obs["counts_per_gene"], bins=50, kde=False)
plt.xlabel("Counts per Gene")
plt.ylabel("Number of Cells")
plt.title("Distribution of Counts per Gene")
plt.show()

In [ ]:
# Scatter plot of total counts vs number of genes
plt.figure(figsize=(6, 4))
sns.scatterplot(x="n_counts", y="n_genes", data=adata.obs, alpha=0.5)
plt.xlabel("Total Counts per Cell")
plt.ylabel("Number of Genes per Cell")
plt.title("Total Counts vs Number of Genes")
plt.show()

In [ ]:
# Make voilen plot of n_counts per sample & n_genes per sample
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.violinplot(x="sample", y="n_counts", data=adata.obs)
plt.xlabel("Sample")
plt.ylabel("Total Counts per Cell")
plt.title("Total Counts per Cell by Sample")

plt.subplot(1, 2, 2)
sns.violinplot(x="sample", y="n_genes", data=adata.obs)
plt.xlabel("Sample")
plt.ylabel("Number of Genes per Cell")
plt.title("Number of Genes per Cell by Sample")

In [ ]:
print("Cells before filtering:", adata.n_obs)
print("Genes before filtering:", adata.n_vars)

In [ ]:
# Check quantiles
adata.obs["n_counts"].quantile([0.01, 0.99])

In [ ]:
# Check quantiles
adata.obs["n_genes"].quantile([0.01, 0.99])

In [ ]:
# Check quantiles
adata.obs["counts_per_gene"].quantile([0.01, 0.99])

In [ ]:
## Compute quantile thresholds
n_counts_low, n_counts_high = adata.obs["n_counts"].quantile([0.01, 0.99])
n_genes_low, n_genes_high = adata.obs["n_genes"].quantile([0.01, 0.99])
complex_low, complex_high = adata.obs["counts_per_gene"].quantile([0.01, 0.99])

# Define mito and ribo thresholds (biologically guided)
mito_threshold = 15        # Common practice: < 10–20%
ribo_threshold = 60        # Dataset dependent, often < 40–50%

# Apply all filters together
adata_filtered = adata[
    (adata.obs["n_counts"] >= n_counts_low) &
    (adata.obs["n_counts"] <= n_counts_high) &
    (adata.obs["n_genes"] >= n_genes_low) &
    (adata.obs["n_genes"] <= n_genes_high) &
    (adata.obs["counts_per_gene"] >= complex_low) &
    (adata.obs["counts_per_gene"] <= complex_high) &
    (adata.obs["pct_counts_mt"] < mito_threshold) &
    (adata.obs["pct_counts_ribo"] < ribo_threshold),
    :
].copy()

In [ ]:
print("Cells after filtering:", adata_filtered.n_obs)
print("Genes after filtering:", adata_filtered.n_vars)
print("Cells removed:", adata.n_obs - adata_filtered.n_obs)

In [ ]:
# Export filtered QC metrics
adata_filtered.obs.to_csv("QC_filtered_table.csv")

In [ ]:
adata_filtered.shape

In [ ]:
adata.shape

In [ ]:
# make voilen plot of n_counts per sample & n_genes per sample after filtering
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
sns.violinplot(x="sample", y="n_counts", data=adata_filtered.obs)
plt.xlabel("Sample")
plt.ylabel("Total Counts per Cell")
plt.title("Total Counts per Cell by Sample (Filtered)")

plt.subplot(1, 2, 2)
sns.violinplot(x="sample", y="n_genes", data=adata_filtered.obs)
plt.xlabel("Sample")
plt.ylabel("Number of Genes per Cell")
plt.title("Number of Genes per Cell by Sample (Filtered)")

In [ ]:
# scatter plot of total counts vs number of genes after filtering
plt.figure(figsize=(6, 4))
sns.scatterplot(x="n_counts", y="n_genes", data=adata_filtered.obs, alpha=0.5)
plt.xlabel("Total Counts per Cell")
plt.ylabel("Number of Genes per Cell")
plt.title("Total Counts vs Number of Genes (Filtered)")
plt.show()

In [ ]:
# Freeze raw counts after QC filtering
adata_filtered.raw = adata_filtered.copy()

In [ ]:
# Check the shape of the raw counts after filtering
adata_filtered.raw.X.shape

In [ ]:
# Normalize counts (library-size normalization)

sc.pp.normalize_total(
    adata_filtered,
    target_sum=1e4
)

In [ ]:
# Total counts per cell (raw)
adata_filtered.obs["total_counts_raw"] = (
    adata_filtered.raw.X.sum(axis=1).A1
)

# Total counts per cell (normalized)
adata_filtered.obs["total_counts_normalized"] = (
    adata_filtered.X.sum(axis=1).A1
)
# Check the first few rows of obs to see the new total counts columns
adata_filtered.obs[["total_counts_raw", "total_counts_normalized"]].head()

In [ ]:
# Log transform normalized data
sc.pp.log1p(adata_filtered)

In [ ]:
# Preserve log-normalized expression
adata_filtered.layers["lognorm"] = adata_filtered.X.copy()

In [ ]:
# Identify highly variable genes (HVGs)
sc.pp.highly_variable_genes(
    adata_filtered,
    flavor="seurat",
    n_top_genes=2000
)

In [ ]:
# Check the first few rows of var to see the new highly variable gene columns
adata_filtered.var.head()


In [ ]:
# Chaeck the number of rows and columns in the adata_filtered.var DataFrame
adata_filtered.var.shape

In [ ]:
# Visualize highly variable genes in normalized mean vs normalized variance plot
sc.pl.highly_variable_genes(adata_filtered)

In [ ]:
# Export HVG statistics to CSV
hvg_stats = adata_filtered.var.copy()
hvg_stats.to_csv("HVG_statistics_full.csv")

In [ ]:
# Extract only the highly variable genes
hvg_only = adata_filtered.var[
    adata_filtered.var["highly_variable"]
]
hvg_only.head()


In [ ]:
# number of rows and columns in the hvg_only DataFrame
hvg_only.shape

In [ ]:
# Export only the highly variable genes statistics to CSV
hvg_only.to_csv("HVG_statistics_HVGonly.csv")

In [ ]:
adata_hvg = adata_filtered[
    :, adata_filtered.var["highly_variable"]
].copy()

In [ ]:
# Convert the sparse matrix of highly variable genes to a DataFrame
expr_df = pd.DataFrame(
    adata_hvg.X,
    index=adata_hvg.obs_names,
    columns=adata_hvg.var_names
)

expr_df.head()


In [ ]:
# Export the expression matrix of highly variable genes to CSV
expr_df.to_csv("HVG_expression_matrix.csv")

In [ ]:
adata_hvg.shape

In [ ]:
# Shape of raw matrix
adata.X.shape

In [ ]:
# shape of matrix after QC
adata_filtered.raw.shape

In [ ]:
# shape of log-normalized matrix
adata_filtered.layers["lognorm"].shape

In [ ]:
# Shape of matrix with only highly variable genes
adata_hvg.shape

In [ ]:
# Scale HVGs
sc.pp.scale(
    adata_hvg,
    max_value=10   # clipping extreme values (important for scRNA-seq)
)

In [ ]:
# Check scaled matrix head
scaled_df = pd.DataFrame(
    adata_hvg.X[:10, :10],
    index=adata_hvg.obs_names[:10],
    columns=adata_hvg.var_names[:10]
)
scaled_df

In [ ]:
# Principal Component Analysis (PCA)

# Step 1: Perform PCA
# Input: scaled HVG matrix (adata_hvg)
# svd_solver="arpack" -> efficient solver for large datasets
sc.tl.pca(
    adata_hvg,
    svd_solver="arpack"
)

In [ ]:
# Step 2: visualize PCA embedding colored by sample
sc.pl.pca(
    adata_hvg,
    color="sample",      # color points by sample metadata
    size=20,             # adjust marker size for clarity
    alpha=0.7            # transparency to see overlapping cells
)

In [ ]:
# Step 3: Compare variance captured by first few PCs vs remaining
explained_variance = adata_hvg.uns["pca"]["variance_ratio"]
cumulative_var = explained_variance.cumsum()
plt.figure(figsize=(6,4))
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o')
plt.xlabel("Principal Component")
plt.ylabel("Cumulative Explained Variance")
plt.title("Cumulative Variance Explained by PCs")
plt.grid(True)
plt.show()

In [ ]:
# Step 4: Inspect PCA output – explained variance
print("Explained variance ratio (first 10 PCs):")
print(adata_hvg.uns["pca"]["variance_ratio"][:10])

In [ ]:
# Step 5: Inspect gene loadings for PC1
# This shows which genes contribute most to PC1
pc1_genes = pd.Series(
    adata_hvg.varm["PCs"][:, 0],  # loadings of PC1
    index=adata_hvg.var_names
).sort_values(ascending=False)
print("Top 10 genes contributing to PC1:")
print(pc1_genes.head(10))

In [ ]:
# Step 6: Export PCA coordinates to CSV
pca_coords = pd.DataFrame(
    adata_hvg.obsm["X_pca"],
    index=adata_hvg.obs_names,
    columns=[f"PC{i+1}" for i in range(adata_hvg.obsm["X_pca"].shape[1])]
)
pca_coords.to_csv("PCA_coordinates.csv")

In [ ]:
# Step 7: Check how many PCs are needed to cover 80% of variance
n_pcs_80 = sum(cumulative_var < 0.80)
print(f"Number of PCs to cover ~80% variance: {n_pcs_80}")

In [ ]:
# Step 8: Top genes per PC
top_genes_per_pc = adata_hvg.varm["PCs"][:, :5]  # first 5 PCs
top_genes_idx = [list(np.argsort(-abs(top_genes_per_pc[:,i]))[:5]) for i in range(5)]
for i, idx in enumerate(top_genes_idx):
    genes = adata_hvg.var_names[idx]
    print(f"Top genes for PC{i+1}: {list(genes)}")

In [ ]:
# Step 9: heatmap of top genes per PC (optional)
top_genes = set()
for idx in top_genes_idx:
    top_genes.update(adata_hvg.var_names[idx])
top_genes = list(top_genes)
sc.pl.heatmap(
    adata_hvg,
    var_names=top_genes,
    groupby="sample",
    use_raw=False,
    show_gene_labels=True
)

In [ ]:
# =============================
# Step 1: Compute the neighborhood graph
# =============================

# Input: scaled HVG matrix (adata_hvg) with PCA already computed
# n_neighbors = 15 -> each cell is connected to its 15 closest neighbors
# n_pcs = 30 -> use first 30 PCs to calculate distances
sc.pp.neighbors(
    adata_hvg,
    n_neighbors=15,
    n_pcs=30
)

# Teaching explanation:
# Each cell is now connected to its 'closest' cells in PC space.
# Think of it as creating a "friendship network" among cells:
#   - Each cell has 15 closest friends (neighbors)
#   - Distance between friends = similarity in gene expression

In [ ]:
# Step 2: Inspect the distance matrix
dist_matrix = adata_hvg.obsp["distances"].toarray()  # convert sparse matrix to dense
dist_matrix.shape  # should be (n_cells, n_cells)

In [ ]:
# Ignore diagonal (distance of a cell to itself)
np.fill_diagonal(dist_matrix, np.nan)

# Maximum distance
max_dist = np.nanmax(dist_matrix)
max_idx = np.unravel_index(np.nanargmax(dist_matrix), dist_matrix.shape)

print(f"Max distance: {max_dist}")
print(f"Between cells: {adata_hvg.obs_names[max_idx[0]]} and {adata_hvg.obs_names[max_idx[1]]}")

In [ ]:
# Minimum distance (closest neighbors)
min_dist = np.nanmin(dist_matrix)
min_idx = np.unravel_index(np.nanargmin(dist_matrix), dist_matrix.shape)

print(f"Min distance: {min_dist}")
print(f"Between cells: {adata_hvg.obs_names[min_idx[0]]} and {adata_hvg.obs_names[min_idx[1]]}")

In [ ]:
# Compute UMAP embedding from the neighborhood graph
sc.tl.umap(
    adata_hvg,
    min_dist=0.3,   # controls cluster compactness (How close cells are allowed to stand together)
    spread=1.0      # controls global separation
)

In [ ]:
# UMAP coordinates (2D embedding)
umap_df = pd.DataFrame(
    adata_hvg.obsm["X_umap"],
    index=adata_hvg.obs_names,
    columns=["UMAP1", "UMAP2"]
)

umap_df.head()

In [ ]:
umap_df.to_csv("UMAP_embeddings_cells.csv")

In [ ]:
# Plain UMAP (structure only)
sc.pl.umap(
    adata_hvg,
    size=20
)

In [ ]:
# UMAP colored by sample identity
sc.pl.umap(
    adata_hvg,
    color="sample",
    size=5
)

In [ ]:
# pip install leidenalg

In [ ]:
import leidenalg

In [ ]:
# Run Leiden clustering on the neighbor graph
sc.tl.leiden(
    adata_hvg,
    resolution=0.3,
    key_added="leiden_0.3"
)

In [ ]:
# Visualize UMAP colored by Leiden clusters
sc.pl.umap(
    adata_hvg,
    color="leiden_0.3",
    legend_loc="on data",
    size=5
)

In [ ]:
# Run Leiden clustering at multiple resolutions
for res in [0.2, 0.4, 0.6]:
    sc.tl.leiden(
        adata_hvg,
        resolution=res,
        key_added=f"leiden_{res}"
    )

In [ ]:
# Visualize UMAP colored by Leiden clusters at multiple resolutions
sc.pl.umap(
    adata_hvg,
    color=["leiden_0.2", "leiden_0.4", "leiden_0.6"],
    wspace=0.4
)

In [ ]:
# Check the number of clusters at each resolution
adata_hvg.obs["leiden_0.3"].value_counts()

In [ ]:
# Export cluster assignments to CSV
adata_hvg.obs["leiden_0.3"].value_counts().to_csv(
    "Cluster_Cell_Counts.csv"
)

In [ ]:
cluster_df = adata_hvg.obs[["sample", "leiden_0.3"]]
cluster_df.to_csv("Cell_Clusters_Leiden_0.3.csv")
cluster_df.head()

In [ ]:
adata_filtered.obs["leiden_0.3"] = adata_hvg.obs["leiden_0.3"]

In [ ]:
# View the first few rows
adata_filtered.obs["leiden_0.3"].head()

In [ ]:
adata_filtered.layers["lognorm"].shape

In [ ]:
# Marker genes for Leiden clusters at resolution 0.3
sc.tl.rank_genes_groups(
    adata_filtered,
    groupby="leiden_0.3",
    method="wilcoxon",
    layer="lognorm",     # uses log1p layer
    pts=True,
    use_raw=False        # ← important
)

In [ ]:
# Export all marker genes for all clusters to CSV
marker_df = sc.get.rank_genes_groups_df(
    adata_filtered,
    group=None           # all clusters
)

marker_df.to_csv("Cluster_Marker_Genes.csv", index=False)
marker_df.head()

In [ ]:
# Define thresholds
LOGFC_CUTOFF = 1
PVALUE_CUTOFF = 0.05
PCT_GROUP_CUTOFF = 0.25
PCT_REF_CUTOFF = 0.25

# Extract the marker table
deg_df = sc.get.rank_genes_groups_df(
    adata_filtered,
    group=None
)

# Apply all gold-standard filters
filtered_deg_df = deg_df[
    (deg_df["pvals_adj"] <= PVALUE_CUTOFF) &
    ((deg_df["logfoldchanges"] >= LOGFC_CUTOFF) |
     (deg_df["logfoldchanges"] <= -LOGFC_CUTOFF)) &
    (deg_df["pct_nz_group"] >= PCT_GROUP_CUTOFF) &
    (deg_df["pct_nz_reference"] <= PCT_REF_CUTOFF)
].copy()

# Save refined markers
filtered_deg_df.to_csv("Refined_Marker_Genes_GoldStandard.csv", index=False)

# View top markers
for cluster in filtered_deg_df["group"].unique():
    print(f"\nCluster {cluster} top markers:")
    display(filtered_deg_df[filtered_deg_df["group"]==cluster].head(5))

In [ ]:
# save refined markers
filtered_deg_df.to_csv("Refined_Marker_Genes.csv", index=False)

In [ ]:
# Select top 5 markers per cluster
top_n = 5
top_markers = (
    filtered_deg_df.groupby("group")
    .apply(lambda x: x.nlargest(top_n, "logfoldchanges"))
    .reset_index(drop=True)
)

# Extract gene names
genes_to_plot = top_markers["names"].unique().tolist()

# Heatmap using Scanpy
sc.pl.heatmap(
    adata_filtered,
    var_names=genes_to_plot,
    groupby="leiden_0.3",
    use_raw=False,       # uses lognorm layer
    layer="lognorm",
    swap_axes=False,     # clusters on y-axis, genes on x-axis
    cmap="viridis",
    show_gene_labels=True,
    figsize=(14,10)
)

In [ ]:
# Dotplot using Scanpy
sc.pl.dotplot(
    adata_filtered,
    var_names=genes_to_plot,
    groupby="leiden_0.3",
    use_raw=False,
    layer="lognorm",
    standard_scale="var",  # scales each gene to [0,1] for comparability
    dot_max=0.5,
    figsize=(14,8)
)

In [ ]:
#pip install celltypist

In [ ]:
# CellTypist for automated cell type annotation
import celltypist
from celltypist import models

In [ ]:
# Show all available models (locally downloaded or from remote index)
#model_list_df = models.models_description(on_the_fly=True)

#print(model_list_df)

In [ ]:
adata_filtered

In [ ]:
# Make a copy of your filtered AnnData
adata_ct = adata_filtered.copy()

In [ ]:
# Use the log-normalized data for CellTypist
adata_ct.X = adata_filtered.layers["lognorm"]

In [ ]:
adata_ct.shape

In [ ]:
from celltypist import models, annotate

# Load the model
model = models.Model.load("Healthy_Mouse_Liver.pkl")

In [ ]:
# Model description
models.models_description()

In [ ]:
# Run annotation with majority voting
predictions = annotate(
    adata_ct,
    model=model,
    majority_voting=True
)

In [ ]:
predictions

In [ ]:
predictions.adata.obs.shape

In [ ]:
# Check the predicted labels
predictions.predicted_labels.majority_voting.value_counts()

In [ ]:
# Remove duplicated indices (if any) from predicted labels
predicted_labels_clean = predictions.predicted_labels[predictions.predicted_labels.index.duplicated(keep='first') == False]

print(predicted_labels_clean.shape)
print(predicted_labels_clean.index.duplicated().sum())

In [ ]:
# Map the predicted labels back to the original AnnData object
adata_ct.obs['celltypist_majority'] = adata_ct.obs_names.map(
    predicted_labels_clean['majority_voting']
)

In [ ]:
# Check the distribution of predicted labels in the original AnnData object
adata_ct.obs['celltypist_majority'].value_counts(dropna=False)

In [ ]:
adata_ct

In [ ]:
# Compute UMAP if not already done
if 'X_umap' not in adata_ct.obsm:
    sc.tl.umap(adata_ct)

# Plot UMAP with legend outside
sc.pl.umap(
    adata_ct,
    color='celltypist_majority',
    legend_loc='right margin',  # place legend outside
    title='UMAP: CellTypist Annotations',
    frameon=False,              # removes frame for cleaner view
    size=4                     # adjust dot size
)

In [ ]:
# Cross-tabulation of clusters vs CellTypist labels
pd.crosstab(
    adata_ct.obs['leiden_0.3'],
    adata_ct.obs['celltypist_majority']
)

In [ ]:
# Select relevant columns
student_csv = adata_ct.obs[['sample', 'sample_id', 'leiden_0.3', 'celltypist_majority']]

# Save
student_csv.to_csv("Annotated_Cells_UMAP.csv")

In [ ]:
# Output directory for DEGs
output_dir = "/media/hdd/nadeem/Anees/DEGs_by_celltype"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
# Define reference sample
reference_sample = "C2W1"

In [ ]:
# Volcano plot thresholds
padj_thresh = 0.05
logfc_thresh = 1

In [ ]:
# List all unique cell types
cell_types = adata_ct.obs['celltypist_majority'].unique()
print("Cell types found:", cell_types)

In [ ]:
# Loop through each cell type
for cell in cell_types:
    print(f"\nProcessing cell type: {cell}")

    # Subset AnnData for the specific cell type
    adata_cell = adata_ct[adata_ct.obs['celltypist_majority'] == cell].copy()

    # Check number of cells per sample
    print(adata_cell.obs['sample'].value_counts())

    # Differential expression analysis
    sc.tl.rank_genes_groups(
        adata_cell,
        groupby='sample',            # phenotype column
        reference=reference_sample,  # control as reference
        method='wilcoxon',           # statistical test
        layer='lognorm',             # use log-normalized data
        pts=True,                    # compute fraction of cells expressing each gene
        use_raw=False                # using processed layer
    )

    # Extract results into a DataFrame
    degs = sc.get.rank_genes_groups_df(adata_cell, group=None)

    # Apply thresholds for significance
    degs['Significant'] = 'Not Sig'
    degs.loc[
        (degs['pvals_adj'] <= padj_thresh) &
        ((degs['logfoldchanges'] >= logfc_thresh) | (degs['logfoldchanges'] <= -logfc_thresh)),
        'Significant'
    ] = 'Significant'

    # Create folder for this cell type
    cell_folder = os.path.join(output_dir, cell.replace(" ", "_"))
    os.makedirs(cell_folder, exist_ok=True)

    # Save raw DEGs and significant DEGs separately
    degs.to_csv(os.path.join(cell_folder, f"{cell}_all_DEGs.csv"), index=False)
    degs_sig = degs[degs['Significant'] == 'Significant']
    degs_sig.to_csv(os.path.join(cell_folder, f"{cell}_significant_DEGs.csv"), index=False)

    print(f"DEGs saved for {cell}: total={len(degs)}, significant={len(degs_sig)}")

    # -------------------------
    # Volcano plot
    plt.figure(figsize=(8,6))
    sns.scatterplot(
        data=degs,
        x='logfoldchanges',
        y=-np.log10(degs['pvals_adj'] + 1e-300),  # avoid -inf
        hue='Significant',
        palette={'Not Sig':'gray', 'Significant':'red'},
        alpha=0.7
    )

    # Add threshold lines
    plt.axvline(x=logfc_thresh, color='blue', linestyle='--', lw=1)
    plt.axvline(x=-logfc_thresh, color='blue', linestyle='--', lw=1)
    plt.axhline(y=-np.log10(padj_thresh), color='green', linestyle='--', lw=1)

    plt.title(f"Volcano Plot: {cell}")
    plt.xlabel("Log2 Fold Change")
    plt.ylabel("-Log10 Adjusted P-value")
    plt.legend(title='Significance')

    # Save volcano plot
    plot_file = os.path.join(cell_folder, f"{cell}_volcano.png")
    plt.tight_layout()
    plt.savefig(plot_file, dpi=300)
    plt.close()

    print(f"Volcano plot saved for {cell}")